In [1]:
#############################
# End-to-End AWS Text Classification System 
# With Multiple Models + Voting + XGBoost + Custom Text Prediction
#############################

### 0. Install & Imports (if needed, adjust for your environment)

# If running locally or Colab, you may need to install xgboost:
# !pip install xgboost

import pandas as pd
import numpy as np
import html
import unicodedata
import re
import string
import nltk

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer

# XGBoost
from xgboost import XGBClassifier

# Download NLTK dependencies
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:

#############################
### 1. Read Data
#############################

df = pd.read_csv("D:/1)debi/project/IMDB Dataset.csv")  # 50K IMDB reviews dataset
print("Data Sample:")
display(df.head())

Data Sample:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


#############################
### 2. Text Preprocessing
#############################

In [3]:


stop_words = set(stopwords.words('english'))

def remove_special_chars(text):
    re1 = re.compile(r'  +')
    x1 = text.lower().replace('#39;', "'").replace('amp;', '&').replace('#146;', "'") \
            .replace('nbsp;', ' ').replace('#36;', '$').replace('\\n', "\n") \
            .replace('quot;', "'").replace('<br />', "\n").replace('\\"', '"') \
            .replace('<unk>', 'u_n').replace(' @.@ ', '.').replace(' @-@ ', '-') \
            .replace('\\', ' \\ ')
    return re1.sub(' ', html.unescape(x1))

def remove_non_ascii(text):
    return unicodedata.normalize('NFKD', text).encode('ascii','ignore').decode('utf-8','ignore')

def to_lowercase(text):
    return text.lower()

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def replace_numbers(text):
    return re.sub(r'\d+', '', text)

def text2words(text):
    return word_tokenize(text)

def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

def lemmatize_words(words):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in words]

def normalize_text(text):
    """Applies all preprocessing steps to text"""
    text = remove_special_chars(text)
    text = remove_non_ascii(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    text = replace_numbers(text)
    words = text2words(text)
    words = remove_stopwords(words)
    words = lemmatize_words(words)
    return ' '.join(words)

# Apply the complete pipeline
df['cleaned_review'] = df['review'].apply(normalize_text)

print("\nCleaned Data Sample:")
display(df[['review','cleaned_review']].head())


Cleaned Data Sample:


,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


In [4]:
#############################
### 3. Feature Extraction (TF-IDF)
#############################

tfidf = TfidfVectorizer(max_features=10000)  # limit feature count
X = tfidf.fit_transform(df['cleaned_review'])

# Convert sentiment to numeric (0 = negative, 1 = positive)
df_one_hot = pd.get_dummies(df, columns=['sentiment'], dtype=int)
y = df_one_hot['sentiment_positive'].values

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state=42)
print("\nShapes after TF-IDF:")
print("Training set:", X_train.shape, "Test set:", X_test.shape)


Shapes after TF-IDF:
Training set: (40000, 10000) Test set: (10000, 10000)


In [5]:
#############################
### 4. Training & Evaluation Helper
#############################

def train_and_evaluate(model, model_name, X_tr, y_tr, X_te, y_te):
    """
    Train a model and print out its accuracy and classification report.
    Returns the fitted model and the predictions on X_te.
    """
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)
    print(f"{model_name} Accuracy: {acc:.4f}")
    print(f"{model_name} Classification Report:")
    print(classification_report(y_te, preds))
    return model, preds

In [6]:
#############################
### 5. Define Multiple Models (including new XGBoost)
#############################

log_reg = LogisticRegression(max_iter=1000, random_state=42)
naive_bayes = MultinomialNB()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='linear', probability=True, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

models = {
    "LogisticRegression": log_reg,
    "NaiveBayes": naive_bayes,
    "RandomForest": rf,
    "SVM": svm,
    "XGBoost": xgb  # newly added algorithm
}

trained_models = {}
predictions = {}

In [7]:
#############################
### 6. Train & Evaluate Each Model Separately
#############################

print("\n--- Training and Evaluating Individual Models ---\n")
for name, model_obj in models.items():
    print("------------------------------------------------")
    fitted_model, y_pred = train_and_evaluate(model_obj, name, 
                                              X_train, y_train, 
                                              X_test, y_test)
    trained_models[name] = fitted_model
    predictions[name] = y_pred
    print("------------------------------------------------\n")


--- Training and Evaluating Individual Models ---

------------------------------------------------
LogisticRegression Accuracy: 0.8921
LogisticRegression Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

------------------------------------------------

------------------------------------------------
NaiveBayes Accuracy: 0.8574
NaiveBayes Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4961
           1       0.86      0.86      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

------

c:\Users\youss\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:45:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Accuracy: 0.8614
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.86      4961
           1       0.85      0.88      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

------------------------------------------------



In [8]:
#############################
### 7. Voting Classifier (Ensemble)
#############################

# We combine all five models in a majority-vote ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('lr', trained_models["LogisticRegression"]),
        ('nb', trained_models["NaiveBayes"]),
        ('rf', trained_models["RandomForest"]),
        ('svm', trained_models["SVM"]),
        ('xgb', trained_models["XGBoost"])
    ],
    voting='hard'  # majority vote
)

voting_clf.fit(X_train, y_train)
ensemble_preds = voting_clf.predict(X_test)
ensemble_acc = accuracy_score(y_test, ensemble_preds)

print("################################################")
print(f"Voting Ensemble Accuracy: {ensemble_acc:.4f}")
print("Voting Ensemble Classification Report:")
print(classification_report(y_test, ensemble_preds))
print("################################################")

c:\Users\youss\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [01:56:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


################################################
Voting Ensemble Accuracy: 0.8928
Voting Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

################################################


In [9]:
#############################
### 8. Testing on a Small Sample of Test Data
#############################

print("\n--- Sample Test Predictions on the Test Set ---")
sample_indices = [0, 1, 2, 3, 4]  # first 5 test examples, for instance
for idx in sample_indices:
    review_vector = X_test[idx]
    true_label = y_test[idx]
    ensemble_prediction = voting_clf.predict(review_vector)[0]
    # Map numeric to label
    pred_label_str = "POSITIVE" if ensemble_prediction == 1 else "NEGATIVE"
    true_label_str = "POSITIVE" if true_label == 1 else "NEGATIVE"
    print(f"Test Sample {idx}: True = {true_label_str}, Predicted = {pred_label_str}")


--- Sample Test Predictions on the Test Set ---
Test Sample 0: True = POSITIVE, Predicted = NEGATIVE
Test Sample 1: True = POSITIVE, Predicted = POSITIVE
Test Sample 2: True = NEGATIVE, Predicted = NEGATIVE
Test Sample 3: True = POSITIVE, Predicted = POSITIVE
Test Sample 4: True = NEGATIVE, Predicted = NEGATIVE


In [10]:
#############################
### 9. Custom Text Prediction
#############################

def predict_custom_text(text, vectorizer, model):
    """
    Preprocess custom text, transform it with the same TF-IDF vectorizer,
    then predict with the provided model. Return predicted sentiment label.
    """
    # Preprocess
    cleaned = normalize_text(text)
    # Vectorize
    transformed = vectorizer.transform([cleaned])
    # Predict
    pred = model.predict(transformed)[0]
    return "POSITIVE" if pred == 1 else "NEGATIVE"

# Allow user to enter custom text (in a real Jupyter environment, you can run this cell and type input)
print("\n--- Custom Text Prediction ---")
user_text = input("Enter a movie review to classify (e.g., 'I loved this movie!'): ")

# Use the voting ensemble as final predictor
prediction_label = predict_custom_text(user_text, tfidf, voting_clf)
print(f"\nYour review is predicted as: {prediction_label}")


--- Custom Text Prediction ---

Your review is predicted as: POSITIVE
